# Lesson 4: Chaining Prompts for Agentic Reasoning

## Automated Claim Triage: From First-Notice to the Right Queue

In this hands-on exercise, you will build a prompt chain that extracts key fields from free-form auto-claim reports, assesses damage severity, and routes each claim to one of several queues — `glass`, `fast_track`, `material_damage`, or `total_loss` — with code-based gate checks at every step.

## Outline:

- Setup
- Sample FNOL (First Notice of Loss) Texts
- Stage I: Information Extraction
- Stage II: Severity Assessment
- Stage III: Queue Routing
- Review Outputs

## Setup

Import necessary libraries and define helper functions, including a mock LLM client, code execution environment, and test runner.

In [71]:
%%time
# Import necessary libraries
# No changes needed in this cell
from openai import OpenAI  # For accessing the OpenAI API
from enum import Enum
import json
from pydantic import BaseModel, Field  # For structured data validation
from typing import List, Literal, Optional

CPU times: total: 0 ns
Wall time: 0 ns


In [72]:
%%time
# Set up LLM credentials
import os
from dotenv import load_dotenv
# Load environment variables from config/.env
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), "config", ".env"))
client = OpenAI(
    base_url="https://openai.vocareum.com/v1",
    api_key=os.getenv("OPENAI_API_KEY"), 
    )

CPU times: total: 1.05 s
Wall time: 1.07 s


In [73]:
%%time
import os
import time
from dataclasses import dataclass
from dotenv import load_dotenv

start_time = time.perf_counter()

load_dotenv()

SUPPORTED_MODELS = {
    "gpt-4o",
    "gpt-4o-mini",
    "gpt-4.1-mini",
    "gpt-4.1-nano",
}


@dataclass(frozen=True)
class Settings:
    api_key: str
    model: str
    max_tokens: int

    @staticmethod
    def load() -> "Settings":
        api_key = os.getenv("OPENAI_API_KEY", "").strip()
        model = os.getenv("OPENAI_MODEL", "gpt-4.1-nano").strip()
        max_tokens_raw = os.getenv("OPENAI_MAX_TOKENS", "1500").strip()

        if not api_key:
            raise ValueError("OPENAI_API_KEY is missing in .env")

        if model not in SUPPORTED_MODELS:
            raise ValueError(
                f"Invalid OPENAI_MODEL='{model}'. "
                f"Allowed values: {', '.join(sorted(SUPPORTED_MODELS))}"
            )

        try:
            max_tokens = int(max_tokens_raw)
        except ValueError:
            raise ValueError(
                f"Invalid OPENAI_MAX_TOKENS='{max_tokens_raw}'. It must be an integer."
            )

        return Settings(
            api_key=api_key,
            model=model,
            max_tokens=max_tokens,
        )

    @property
    def masked_api_key(self) -> str:
        """
        Shows only beginning and end of the API key.
        Example: sk-proj-abc...xyz123
        """
        if len(self.api_key) <= 10:
            return "*" * len(self.api_key)
        return f"{self.api_key[:8]}...{self.api_key[-6:]}"


settings = Settings.load()

API_KEY = settings.api_key
MODEL = settings.model
MAX_TOKENS = settings.max_tokens

# Your main code starts here
print("Running application...")
# Example: actual work
time.sleep(1.2)

end_time = time.perf_counter()
total_time = end_time - start_time

print("\n========== CONFIG / RUN SUMMARY ==========")
print(f"OpenAI API Key : {settings.masked_api_key}")
print(f"Model Selected : {MODEL}")
print(f"Max Tokens     : {MAX_TOKENS}")
print(f"Total Time     : {total_time:.2f} seconds")
print("==========================================")

Running application...

========== CONFIG / RUN SUMMARY ==========
OpenAI API Key : sk-proj-...IIazsA
Model Selected : gpt-4o
Max Tokens     : 1500
Total Time     : 1.20 seconds
CPU times: total: 0 ns
Wall time: 1.21 s


In [74]:
%%time
# Helper function to call the OpenAI chat completion API

#def get_completion(messages, model=MODEL, max_tokens=MAX_TOKENS):
#    """Send messages to the LLM and return the response content."""
#    response = client.chat.completions.create(
#        model=model,
#        messages=messages,
#        max_tokens=max_tokens,
#        temperature=0,
#    )
#    return response.choices[0].message.content

def get_completion(messages=None, system_prompt=None, user_prompt=None, model=MODEL):
    """
    Function to get a completion from the OpenAI API.
    Args:
        system_prompt: The system prompt
        user_prompt: The user prompt
        model: The model to use (default is gpt-4.1-mini)
    Returns:
        The completion text
    """

    messages = list(messages)
    if system_prompt:
        messages.insert(0, {"role": "system", "content": system_prompt})
    if user_prompt:
        messages.append({"role": "user", "content": user_prompt})
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.7,
    )
    return response.choices[0].message.content


CPU times: total: 0 ns
Wall time: 0 ns


## Sample FNOL (First Notice of Loss) Texts
Let's define a few sample First Notice of Loss (FNOL) texts to process through our chain.

In [75]:
%%time
# Define sample FNOL texts
# TODO: [Optional] Add more sample FNOL texts to test various scenarios

sample_fnols = [
    """
    Claim ID: C001
    Customer: John Smith
    Vehicle: 2018 Toyota Camry
    Incident: While driving on the highway, a rock hit my windshield and caused a small chip
    about the size of a quarter. No other damage was observed.
    """,
    """
    Claim ID: C002
    Customer: Sarah Johnson
    Vehicle: 2020 Honda Civic
    Incident: I was parked at the grocery store and returned to find someone had hit my car and
    dented the rear bumper and taillight. The taillight is broken and the bumper has a large dent.
    """,
    """
    Claim ID: C003
    Customer: Michael Rodriguez
    Vehicle: 2022 Ford F-150
    Incident: I was involved in a serious collision at an intersection. The front of my truck is
    severely damaged, including the hood, bumper, radiator, and engine compartment. The airbags
    deployed and the vehicle is not drivable.
    """,
    """
    Claim ID: C004
    Customer: Emma Williams
    Vehicle: 2019 Subaru Outback
    Incident: My car was damaged in a hailstorm. There are multiple dents on the hood, roof, and
    trunk. The side mirrors were also damaged and one window has a small crack.
    """,
    """
    Claim ID: C005
    Customer: David Brown
    Vehicle: 2021 Tesla Model 3
    Incident: Someone keyed my car in the parking lot. There are deep scratches along both doors
    on the driver's side.
    """,
   


]

CPU times: total: 0 ns
Wall time: 0 ns


## Stage I: Information Extraction
In this stage, we'll create a prompt that extracts structured information from free-form FNOL text.

In [76]:
%%time
# Define the ClaimInformation model and extraction system prompt


class ClaimInformation(BaseModel):
    claim_id: str = Field(..., min_length=2, max_length=10)
    name: str = Field(..., min_length=2, max_length=100)
    vehicle: str = Field(..., min_length=2, max_length=100)
    loss_desc: str = Field(..., min_length=10, max_length=500)
    damage_area: List[
        Literal[
            "windshield",
            "front",
            "rear",
            "side",
            "roof",
            "hood",
            "door",
            "bumper",
            "fender",
            "quarter panel",
            "trunk",
            "glass",
        ]
    ] = Field(..., min_length=1)


info_extraction_system_prompt = """
You are an expert data extraction AI specializing in insurance claims. Your task is to extract key information from First Notice of Loss (FNOL) reports.

Format your response as a valid JSON object with the following keys:
- claim_id (str): The claim ID (e.g. "C001")
- name (str): The customer's full name
- vehicle (str): The year, make, and model of the vehicle
- loss_desc (str): A concise summary of the incident and damage described
- damage_area (list[str]): One or more affected areas from ONLY these values: "windshield", "front", "rear", "side", "roof", "hood", "door", "bumper", "fender", "quarter panel", "trunk", "glass"

Rules:
- Extract information exactly as stated in the report; do not infer or fabricate details.
- damage_area must contain at least one item and only use the allowed values listed above.
- loss_desc should be between 10 and 500 characters.

Only respond with the JSON object, nothing else.
"""

CPU times: total: 15.6 ms
Wall time: 5.13 ms


In [77]:
%%time
# Define a gate check function and claim extraction function


def gate1_validate_claim_info(claim_info_json: str) -> ClaimInformation:
    """
    Gate 1: Validates claim information extracted from FNOL text.
    Returns validated ClaimInformation object or raises validation error.
    """
    try:
        # Parse the JSON string
        claim_info_dict = json.loads(claim_info_json)
        # Validate with Pydantic model
        validated_info = ClaimInformation(**claim_info_dict)
        return validated_info
    except Exception as e:
        raise ValueError(f"Gate 1 validation failed: {str(e)}")


def extract_claim_info(fnol_text):
    """
    Stage 1: Extract structured information from FNOL text
    """
    messages = [
        {"role": "system", "content": info_extraction_system_prompt},
        {"role": "user", "content": fnol_text},
    ]

    response = get_completion(messages=messages)

    # Gate check: validate the extracted information
    try:
        validated_info = gate1_validate_claim_info(response)
        return validated_info
    except ValueError as e:
        print(f"Gate 1 failed: {e}")
        return None

CPU times: total: 0 ns
Wall time: 0 ns


In [78]:
# Run the claim extraction function on the sample FNOLs

extracted_claim_info_items = [
    extract_claim_info(fnol_text) for fnol_text in sample_fnols
]
extracted_claim_info_items

BadRequestError: Error code: 400 - {'error': {'code': None, 'message': 'This key was not found. Please check key was inputed correctly.', 'param': None, 'type': 'invalid_request_error'}}

In [ ]:
%%time
# Define a system prompt for severity assessment according to the provided SeverityAssessment class


class SeverityAssessment(BaseModel):
    severity: Literal["Minor", "Moderate", "Major"]
    est_cost: float = Field(..., gt=0)


severity_assessment_system_prompt = """
You are an experienced auto damage claims adjuster. Based on the claim information, particularly the loss_desc and damage_area, classify the damage as "Minor", "Moderate", or "Major". Also provide an estimated repair cost (est_cost) as a float.

Use the following guidelines for severity and typical costs:
- "Minor": Small dents, scratches, glass chips. Cost range: $100 up to $1,000.
- "Moderate": Single panel damage, bumper replacement, door damage. Cost range: more than $1,000 up to $5,000.
- "Major": Structural damage, multiple panel replacement, engine/drivetrain issues, total loss candidates. Cost range: more than $5,000 up to $50,000.

Format your response as a valid JSON object with the following keys:
- severity (str): One of "Minor", "Moderate", or "Major"
- est_cost (float): The estimated repair cost in dollars (must be within the range for the chosen severity)

Only respond with the JSON object, nothing else.
"""

In [ ]:
%%time
# Define gate check and severity assessment functions


def gate2_cost_range_ok(severity_json: str) -> SeverityAssessment:
    """
    Gate 2: Validates that the estimated cost is within reasonable range for the severity.
    Returns validated SeverityAssessment object or raises validation error.
    """
    try:
        severity_dict = json.loads(severity_json)
        validated_severity = SeverityAssessment(**severity_dict)

        if (
            validated_severity.severity == "Minor"
            and (validated_severity.est_cost < 100 or validated_severity.est_cost > 1000)
        ):
            raise ValueError(
                f"Minor damage should cost between $100-$1000, got ${validated_severity.est_cost}"
            )
        elif (
            validated_severity.severity == "Moderate"
            and (validated_severity.est_cost < 1000 or validated_severity.est_cost > 5000)
        ):
            raise ValueError(
                f"Moderate damage should cost between $1000-$5000, got ${validated_severity.est_cost}"
            )
        elif (
            validated_severity.severity == "Major"
            and (validated_severity.est_cost < 5000 or validated_severity.est_cost > 50000)
        ):
            raise ValueError(
                f"Major damage should cost between $5000-$50000, got ${validated_severity.est_cost}"
            )

        return validated_severity
    except Exception as e:
        raise ValueError(f"Gate 2 validation failed: {str(e)}")


def assess_severity(claim_info: ClaimInformation) -> Optional[SeverityAssessment]:
    """
    Stage 2: Assess severity based on damage description
    """
    claim_info_json = claim_info.model_dump_json()

    messages = [
        {"role": "system", "content": severity_assessment_system_prompt},
        {"role": "user", "content": claim_info_json},
    ]

    response = get_completion(messages=messages)

    try:
        validated_severity = gate2_cost_range_ok(response)
        return validated_severity
    except ValueError as e:
        print(f"Gate 2 failed: {e}. Response: {response}")
        return None

In [ ]:
%%time
# Run the claim extraction function on the sample data
# No updates needed in this cell

severity_assessment_items = [
    assess_severity(item) for item in extracted_claim_info_items
]

severity_assessment_items

In [ ]:
%%time
# Define a system prompt for claim routing according to the provided ClaimRouting class


class ClaimRouting(BaseModel):
    claim_id: str
    queue: Literal["glass", "fast_track", "material_damage", "total_loss"]
    priority: int = Field(..., ge=1, le=5)


queue_routing_system_prompt = """
You are an AI claim routing system. Based on the claim information and severity assessment, assign the claim to one of the following queues: "glass", "fast_track", "material_damage", or "total_loss".

Use these routing rules:
- "glass": Minor damage involving ONLY glass (windshield, windows)
- "fast_track": Other Minor damage (scratches, small dents, cosmetic issues)
- "material_damage": All Moderate damage
- "total_loss": All Major damage

Also assign a priority from 1 (highest) to 5 (lowest) based on the overall situation:
- 1 (highest): Safety issues, customer stranded, vehicle not drivable
- 2: Significant but contained damage, vehicle drivable
- 3: Standard claims with moderate impact
- 4: Minor issues, no mobility impact
- 5 (lowest): Cosmetic only, no functional impact

Format your response as a valid JSON object with the following keys:
- claim_id (str): The claim ID from the input
- queue (str): One of "glass", "fast_track", "material_damage", or "total_loss"
- priority (int): A value from 1 to 5

Only respond with the JSON object, nothing else.
"""

In [ ]:
%%time
# Define gate check and routing functions


def gate3_validate_routing(routing_json: str) -> ClaimRouting:
    """
    Gate 3: Validates that the claim is routed to a valid queue.
    Returns validated ClaimRouting object or raises validation error.
    """
    try:
        routing_dict = json.loads(routing_json)
        validated_routing = ClaimRouting(**routing_dict)
        return validated_routing
    except Exception as e:
        raise ValueError(f"Gate 3 validation failed: {str(e)}")


def route_claim(
    claim_info: ClaimInformation, severity_assessment: Optional[SeverityAssessment]
) -> Optional[ClaimRouting]:
    """
    Stage 3: Route claim to appropriate queue
    """
    if severity_assessment is None:
        return None

    routing_input = {
        "claim_info": claim_info.model_dump(),
        "severity_assessment": severity_assessment.model_dump(),
    }

    messages = [
        {"role": "system", "content": queue_routing_system_prompt},
        {"role": "user", "content": json.dumps(routing_input)},
    ]

    response = get_completion(messages=messages)

    try:
        validated_routing = gate3_validate_routing(response)
        return validated_routing
    except ValueError as e:
        print(f"Gate 3 failed: {e}. Response: {response}")
        return None

In [ ]:
%%time
# Run the routing function on the sample data

routed_claim_items = [
    route_claim(claim, severity_assessment)
    for claim, severity_assessment in zip(
        extracted_claim_info_items, severity_assessment_items
    )
]

routed_claim_items

In [ ]:
%%time
# Review outputs in a pandas dataframe

import pandas as pd

records = []
for claim, severity_assessment, routed_claim in zip(
    extracted_claim_info_items, severity_assessment_items, routed_claim_items
):
    # Skip records where any stage returned None
    if claim is None or severity_assessment is None or routed_claim is None:
        continue
    record = {}
    record.update(claim.model_dump())
    record.update(severity_assessment.model_dump())
    record.update(routed_claim.model_dump())
    records.append(record)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame(records)

df

## 8. Reflection & Transfer
Reflect on the effectiveness of chaining prompts for this task:

1. Prompt Chain Architecture:
* How does breaking down the task into stages affect the performance?
* What are the benefits of having gate checks between stages?
* How could the chain design be improved?

2. Error Handling:
* What types of errors might occur at each stage?
* How could we make the chain more robust against errors?
* What would a good fallback strategy look like?

3. Scalability:
* How well would this approach scale to handle more complex claims?
* What challenges might arise when processing thousands of claims?
* How could the prompt chain be optimized for efficiency?

4. Transfer to Other Domains:
* How could this prompt chaining approach be applied to other domains?
* What general principles of prompt chaining can we extract from this exercise?

## Summary

🎉 Congratulations! 🎉 You've built an impressive prompt chain system for insurance claims!
You transformed messy FNOL text into structured data, assessed damage severity, and routed claims to the right queues, all with robust gate checks! 🚀✨

Remember:

- 🔗 Chained prompts break complex tasks into manageable steps
- 🛡️ Gate checks prevent error cascades
- 🧠 Having specialized prompts helps keep code focused and maintainable

You've mastered a powerful pattern for countless business processes! 🏆
Amazing work on your agentic reasoning system! 💯